# 阶段1 · 步骤01：检查 GO 与 MELA 基因程序

步骤01建立小鼠胚胎项目的资源底表：核对本地 GO、本地 MELA 作者程序及版本，导出 BP 术语和程序成员。

本步骤不启动 Spark 或模型推理。小鼠 ID 映射、GO 注释、富集和文献证据将在后续步骤完成；输出会明确记录这些待办。

请选择 **Python (pathak)** 内核。


## 1. 进入项目并读取配置

配置文件保留物种、发育范围、路径和输入校验值。保留原始基因写法，后续用官方 ID 映射解决别名。


In [1]:
from pathlib import Path
import sys
import os
import json
import hashlib
from datetime import datetime, timezone

PROJECT_ROOT = Path("/root/gpufree-data/PATHAK")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathak_runtime import configure
configure()

import pandas as pd
import obonet

with (PROJECT_ROOT / "config/project.json").open(encoding="utf-8") as handle:
    config = json.load(handle)

GO_PATH = PROJECT_ROOT / config["ontology"]["path"]
PROGRAMS_PATH = PROJECT_ROOT / config["mela_programs"]["path"]
OUTPUT_DIR = PROJECT_ROOT / config["output_directory"]

print("Python:", sys.executable)
print("项目:", config["project"])
print("物种:", config["species"])
print("GO:", GO_PATH)
print("MELA:", PROGRAMS_PATH)


Python: /opt/conda/envs/pathak/bin/python
项目: PATHAK-embryo
物种: {'scientific_name': 'Mus musculus', 'taxon_id': 10090}
GO: /root/gpufree-data/PATHAK/data/go/go.obo
MELA: /root/gpufree-data/PATHAK/data/raw/mela/annotated_programs.csv


## 2. 校验输入

校验值用于判断是否仍在分析同一份资源。GO 文件保留在永久数据目录中。MELA 程序表来自 CPU 的已保存作者数据。


In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for path in [GO_PATH, PROGRAMS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(str(path))

go_sha256 = sha256_file(GO_PATH)
programs_sha256 = sha256_file(PROGRAMS_PATH)
if programs_sha256 != config["mela_programs"]["sha256"]:
    raise ValueError("MELA 输入校验值变化，请先核对来源并更新版本记录。")

print("GO SHA-256:", go_sha256)
print("MELA SHA-256:", programs_sha256)


GO SHA-256: d3593751d885ca160b2ab7baf6c7eccd88ca3c4599f79436c674bad661095ff0
MELA SHA-256: 1489862271bbeb10a1dc9f0eaf7355d131fdfdfdb8455a55b3e2030dca9523c4


## 3. 读取 GO 并严格选择 BP

按 namespace 选择 Biological Process。沿所有关系边追溯 BP 根节点会混入其他分支，因此本步骤不使用这种筛选方式。

保留 OBO 定义和同义词原文，避免在资源整理阶段丢失信息。根术语也保留在资源表中并打标记，后续检索时排除。


In [3]:
graph = obonet.read_obo(str(GO_PATH))
go_version = graph.graph.get("data-version", "unknown")
namespace_counts = pd.Series(
    [attrs.get("namespace", "unknown") for _, attrs in graph.nodes(data=True)]
).value_counts()

bp_rows = []
for go_id, attrs in graph.nodes(data=True):
    if attrs.get("namespace") != config["ontology"]["namespace"]:
        continue
    bp_rows.append({
        "GO": go_id,
        "Name": attrs.get("name", ""),
        "Namespace": attrs["namespace"],
        "Definition_OBO": attrs.get("def", ""),
        "Synonyms_OBO": json.dumps(attrs.get("synonym", []), ensure_ascii=False),
        "is_namespace_root": go_id == config["ontology"]["root_id"],
    })

bp_terms = pd.DataFrame(bp_rows).sort_values("GO").reset_index(drop=True)
assert bp_terms["GO"].is_unique
assert bp_terms["Namespace"].eq("biological_process").all()

print("GO 版本:", go_version)
print("全图节点数:", graph.number_of_nodes())
print("BP 术语数（含根）:", len(bp_terms))
display(namespace_counts.rename_axis("namespace").to_frame("nodes"))
display(bp_terms.head())


GO 版本: releases/2026-07-26
全图节点数: 38092
BP 术语数（含根）: 23974


,nodes
namespace,
biological_process,23974
molecular_function,10041
cellular_component,4077


,GO,Name,Namespace,Definition_OBO,Synonyms_OBO,is_namespace_root
0,GO:0000001,mitochondrion inheritance,biological_process,"""The distribution of mitochondria, including t...","[""\""mitochondrial inheritance\"" EXACT []""]",False
1,GO:0000011,vacuole inheritance,biological_process,"""The distribution of vacuoles into daughter ce...",[],False
2,GO:0000012,single strand break repair,biological_process,"""The repair of single strand breaks in DNA. Re...",[],False
3,GO:0000017,alpha-glucoside transport,biological_process,"""The directed movement of alpha-glucosides int...",[],False
4,GO:0000018,regulation of DNA recombination,biological_process,"""Any process that modulates the frequency, rat...",[],False


## 4. 拆解 MELA 作者程序

一个程序包含一组原始基因名称。这里把逗号分隔的成员展开成每行一个程序—基因，并核对作者 size 字段；不统一大写、不自动替换历史基因名。

active_in、name、description 保持作者原文。它们不是新的逐细胞注释或独立实验验证。


In [4]:
programs = pd.read_csv(PROGRAMS_PATH, dtype=str, keep_default_na=False)
required = {"program", "genes", "size", "active_in", "name", "description"}
missing_columns = required - set(programs.columns)
if missing_columns:
    raise ValueError("缺少列: " + ", ".join(sorted(missing_columns)))
if len(programs) != config["mela_programs"]["expected_programs"]:
    raise ValueError("程序数量与固定版本不符。")
assert programs["program"].is_unique

member_rows = []
for row in programs.itertuples(index=False):
    genes = [item.strip() for item in row.genes.split(",") if item.strip()]
    for position, gene in enumerate(genes, start=1):
        member_rows.append({
            "program": row.program,
            "gene_raw": gene,
            "member_position": position,
        })

membership = pd.DataFrame(member_rows)
parsed_sizes = membership.groupby("program").size()
unique_sizes = membership.groupby("program")["gene_raw"].nunique()
program_audit = programs[["program", "name", "size"]].copy()
program_audit["declared_size"] = pd.to_numeric(program_audit["size"], errors="raise")
program_audit["parsed_gene_entries"] = program_audit["program"].map(parsed_sizes).fillna(0).astype(int)
program_audit["unique_gene_names"] = program_audit["program"].map(unique_sizes).fillna(0).astype(int)
program_audit["matches_declared_size"] = (
    program_audit["declared_size"] == program_audit["parsed_gene_entries"]
)
duplicate_members = membership[membership.duplicated(["program", "gene_raw"], keep=False)]

print("程序数:", len(programs))
print("程序—基因条目:", len(membership))
print("不同原始基因名称:", membership["gene_raw"].nunique())
print("size 不一致程序:", int((~program_audit["matches_declared_size"]).sum()))
print("重复成员条目:", len(duplicate_members))
display(program_audit.head(10))
display(membership.head(10))


程序数: 70
程序—基因条目: 1322
不同原始基因名称: 1322
size 不一致程序: 0
重复成员条目: 0


,program,name,size,declared_size,parsed_gene_entries,unique_gene_names,matches_declared_size
0,P1,Posterior axial elongation,56,56,56,56,True
1,P2,Trunk hox ap patterning,26,26,26,26,True
2,P3,Posterior hox patterning,16,16,16,16,True
3,P4,Mesoderm pcp/ecm remodeling,7,7,7,7,True
4,P5,Caudal axial patterning,6,6,6,6,True
5,P6,Osr1+ mesoderm identity,6,6,6,6,True
6,P7,Ra progenitor maintenance,5,5,5,5,True
7,P8,Psm fgf9-ecm signaling,5,5,5,5,True
8,P9,Cns neural progenitor,155,155,155,155,True
9,P10,Neuronal differentiation,58,58,58,58,True


,program,gene_raw,member_position
0,P1,9530020I12Rik,1
1,P1,Tbx6,2
2,P1,Fgf17,3
3,P1,Hoxa9,4
4,P1,Hoxc9,5
5,P1,Hoxaas3,6
6,P1,Hoxb9,7
7,P1,Cdx2,8
8,P1,Hes7,9
9,P1,Hoxc8,10


## 5. 保存底表和未完成资源

小鼠 ID 表、GO 注释还未准备时，明确输出 pending。出现待办不代表前面的本地检查失败，也不表示小鼠功能注释已经完成。


In [5]:
pending_rows = []
for resource in config["pending_mouse_resources"]:
    path = PROJECT_ROOT / resource["planned_path"]
    pending_rows.append({
        "resource": resource["resource"],
        "path": str(path),
        "file_exists": path.is_file(),
        "validation_status": "pending_validation" if path.is_file() else "not_prepared",
    })
pending_resources = pd.DataFrame(pending_rows)
display(pending_resources)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bp_terms.to_csv(OUTPUT_DIR / "go_bp_terms.tsv", sep="\t", index=False)
membership.to_csv(OUTPUT_DIR / "mela_program_gene_membership.tsv", sep="\t", index=False)
program_audit.to_csv(OUTPUT_DIR / "mela_program_audit.tsv", sep="\t", index=False)
pending_resources.to_csv(OUTPUT_DIR / "pending_resources.tsv", sep="\t", index=False)

audit = {
    "project": config["project"],
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "species": config["species"],
    "go_version": go_version,
    "go_sha256": go_sha256,
    "go_nodes": graph.number_of_nodes(),
    "bp_terms_including_root": len(bp_terms),
    "mela_programs_sha256": programs_sha256,
    "mela_programs": len(programs),
    "membership_rows": len(membership),
    "unique_raw_gene_names": int(membership["gene_raw"].nunique()),
    "program_size_mismatches": int((~program_audit["matches_declared_size"]).sum()),
    "duplicate_member_rows": len(duplicate_members),
    "pending_resources": pending_rows,
    "mouse_gene_mapping_completed": False,
    "go_enrichment_completed": False,
    "text_embeddings_generated": False,
    "expression_or_lineage_analysis_completed": False,
}
with (OUTPUT_DIR / "resource_audit.json").open("w", encoding="utf-8") as handle:
    json.dump(audit, handle, ensure_ascii=False, indent=2)
print("输出目录:", OUTPUT_DIR)


,resource,path,file_exists,validation_status
0,mouse_gene_identifiers,/root/gpufree-data/PATHAK/data/reference/mouse...,False,not_prepared
1,mouse_go_annotations,/root/gpufree-data/PATHAK/data/reference/mouse...,False,not_prepared


输出目录: /root/gpufree-data/PATHAK/results/01_resources


## 步骤02：小鼠基因标识与 GO 注释

阶段1的后续文件约定为 `notebooks/stage1/02_mouse_ID_GO_annotations.ipynb`；本次仅整理目录与命名，步骤02内容后续编写。

准备小鼠官方基因标识和 GO 注释，建立原始名称到稳定 ID 的映射，并保留歧义与未映射清单。之后以明确的候选基因背景开展 70 个程序的常规 GO 分析。

正式文本检索前还需固定 BioBERT 编码方式，复核单句/批量一致性和 GO—向量行对应，再通过人工核查校准候选关系。
